### from collections.abc import Mapping

직접 Mapping ABC를 상속해서 커스텀 매핑을 만드는 일은 실무에서 흔하지 않습니다.
대부분의 경우 dict, defaultdict, ChainMap 등으로 충분히 해결되기 때문입니다.


### 사례 1 — 타입 힌트 / isinstance 검사용 (가장 흔한 용도)
직접 상속보다, "이 함수가 dict뿐 아니라 모든 매핑을 받는다" 는 것을 표현할 때 씁니다.

In [ ]:
from collections.abc import Mapping

# ❌ 너무 좁음 — dict만 받음
def process(data: dict): ...

# ✅ 실무 권장 — dict, OrderedDict, ChainMap 등 모두 수용
def process(data: Mapping[str, int]): ...

# isinstance 검사도 동일
def normalize(config):
    if not isinstance(config, Mapping):
        raise TypeError("config must be a mapping type")

### 사례 2 — 읽기 전용 래퍼 (보안/불변성 강제)
설정 객체나 공유 상태를 절대 수정 못 하게 막을 때 씁니다.
MutableMapping이 아닌 Mapping만 상속하면 __setitem__이 없으므로 수정이 원천 차단됩니다.

In [ ]:
from collections.abc import Mapping

class ReadOnlyDict(Mapping):
    def __init__(self, data: dict):
        self._data = dict(data)     # 딕셔너리 형태로 변경

    def __getitem__(self, key):
        return self._data[key]

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"ReadOnlyDict({self._data})"

# 실제 사용
set_base = {"db_host": "localhost", "port": 5432}
config = ReadOnlyDict(set_base)

print(config["db_host"])   # "localhost"


localhost


In [5]:
# 매핑된 딕셔너리를 직접 수정 불가함 (260519)

config["port"] = 9999      # ❌ TypeError 발생 — 수정 불가

TypeError: 'ReadOnlyDict' object does not support item assignment

In [ ]:
set_base["port"] = 9999 # 원본 set을 수정
set_base

{'db_host': 'localhost', 'port': 9999}

In [ ]:
config = ReadOnlyDict(set_base)  #매핑을 실행해야 갱신이 되는데... (260519)
config["port"]

9999

In [12]:
# 읽기 전용 MappingProxyType

from types import MappingProxyType

d = set_base
d_proxy = MappingProxyType(d)
d_proxy

mappingproxy({'db_host': 'localhost', 'port': 9999})

In [ ]:
d_proxy[0] # 교재 내용과 불일치 (260519)

KeyError: 0

### 사례 3 — 지연 로딩 매핑 (DB / 파일 / API 연동)
전체 데이터를 메모리에 올리지 않고, 키 접근 시점에만 데이터를 가져오는 패턴입니다.

In [29]:
import json

config_json = json.loads('''{"database": {
    "host": "localhost",
    "port": 5432,
    "name": "myapp_db",
    "user": "admin",
    "password": "secret123",
    "pool_size": 10
  },
  "cache": {
    "backend": "redis",
    "host": "127.0.0.1",
    "port": 6379,
    "ttl": 300
  },
  "api": {
    "base_url": "https://api.myapp.com",
    "version": "v2",
    "timeout": 30,
    "retry": 3
  },
  "logging": {
    "level": "INFO",
    "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    "output": "logs/app.log"
  },
  "features": {
    "enable_signup": true,
    "enable_oauth": true,
    "maintenance_mode": false,
    "max_upload_mb": 50
  }
}
''')

In [34]:
from collections.abc import Mapping
import json

class JSONFileMapping(Mapping):
    """대용량 JSON 파일을 전부 로드하지 않고 키 접근 시에만 읽기"""

    #def __init__(self, filepath: str):
    def __init__(self, data: dict):
        #self._path = filepath
        #self._cache = {}
        self._cache = data

    #def _load(self):
    #    if not self._cache:
    #        with open(self._path) as f:
    #            self._cache = json.load(f)

    def __getitem__(self, key):
        #self._load()
        return self._cache[key]

    def __iter__(self):
        #self._load()
        return iter(self._cache)

    def __len__(self):
        #self._load()
        return len(self._cache)

# 파일을 즉시 읽지 않음
# config = JSONFileMapping("config.json")
config = JSONFileMapping(config_json)

# 이 시점에 처음 파일 접근
print(config["database"])

# 실무 맥락: 대형 설정 파일, 
# S3/GCS 원격 설정, Redis 해시를 dict처럼 쓰고 싶을 때 이 패턴을 활용합니다.

{'host': 'localhost', 'port': 5432, 'name': 'myapp_db', 'user': 'admin', 'password': 'secret123', 'pool_size': 10}


사실 현재 코드에서 __init__에서 self._cache = data로 dict를 통째로 받는 순간 
이미 메모리에 올라가 있습니다. 지연 로딩이 아닙니다.
: 지연 로딩의 의미는 파일 버전에서만 성립합니다. 변수로 전달하는 순간 그 개념은 사라집니다.

### 사례 4 — 계산형 매핑 (값을 저장하지 않고 즉석 계산)
실제로 값을 저장하지 않고, 키를 받아 즉석에서 값을 계산해서 반환합니다.

In [36]:
from collections.abc import Mapping

class FXRateMapping(Mapping):
    """환율 API 결과를 dict처럼 접근 가능하게 래핑"""

    def __init__(self, base: str, rates: dict):
        self._base = base
        self._rates = rates  # {"USD": 1.0, "KRW": 1350.0, ...}

    def __getitem__(self, currency: str) -> float:
        if currency not in self._rates:
            raise KeyError(f"Unknown currency: {currency}")
        return self._rates[currency]

    def convert(self, amount: float, to: str) -> float:
        return amount * self[to]

    def __iter__(self):
        return iter(self._rates)

    def __len__(self):
        return len(self._rates)

rates = FXRateMapping("USD", {"USD": 1.0, "KRW": 1350.0, "EUR": 0.92})
print(rates["KRW"])                  # 1350.0
print(rates.convert(100, "KRW"))     # 135000.0

# 기존 Mapping 인터페이스 그대로 활용 가능
print("EUR" in rates)                # True
print(dict(rates))                   # 일반 dict로 변환도 가능

1350.0
135000.0
True
{'USD': 1.0, 'KRW': 1350.0, 'EUR': 0.92}


In [37]:
rates["USD"]

1.0

직접 구현보다는 Mapping을 타입으로 활용하는 것이 실무에서 압도적으로 많고, 
접 상속이 필요한 순간은 "기존 dict로는 표현할 수 없는 동작"이 명확히 있을 때로 한정됩니다.

### 파일 버전에서 지연로딩을 증명하는 방법

In [39]:
import tracemalloc  # 메모리 추적
import json
from collections.abc import Mapping

class JSONFileMapping(Mapping):
    def __init__(self, filepath: str):
        self._path = filepath
        self._cache = {}

    def _load(self):
        if not self._cache:
            with open(self._path) as f:
                self._cache = json.load(f)

    def __getitem__(self, key):
        self._load()
        return self._cache[key]

    def __iter__(self):
        self._load()
        return iter(self._cache)

    def __len__(self):
        self._load()
        return len(self._cache)


tracemalloc.start()

# ── 객체 생성 직후 ──────────────────────────────
config = JSONFileMapping("config.json")

snap1 = tracemalloc.take_snapshot()
size1 = sum(s.size for s in snap1.statistics("lineno"))
print(f"[생성 직후]  메모리: {size1:,} bytes")
print(f"[생성 직후]  _cache 내용: {config._cache}")  # {} 비어있음

# ── 첫 키 접근 후 ───────────────────────────────
_ = config["database"]

snap2 = tracemalloc.take_snapshot()
size2 = sum(s.size for s in snap2.statistics("lineno"))
print(f"\n[키 접근 후] 메모리: {size2:,} bytes")
print(f"[키 접근 후] _cache 내용: {list(config._cache.keys())}")  # 데이터 존재
print(f"\n차이: +{size2 - size1:,} bytes 증가")

[생성 직후]  메모리: 1,167 bytes
[생성 직후]  _cache 내용: {}

[키 접근 후] 메모리: 6,985 bytes
[키 접근 후] _cache 내용: ['database', 'cache', 'api', 'logging', 'features']

차이: +5,818 bytes 증가


In [ ]:
# 객체 생성 직후와 첫 키 접근 후에 결과를 비교함
# 객체 생성 직후에는 전체 데이터를 메모리에 올리지 않고, 
# 키 접근 시점에만 데이터를 가져오는 패턴임을 알 수 있다. (260519)

# _cache가 {}로 비어있다가 키 접근 후 채워지는 것, 
# 그리고 그 시점에 메모리가 증가하는 것으로 지연 로딩을 직접 확인할 수 있습니다.

In [ ]:
d = dict(a=10, b=20, c=30)  # 키워드 인수로 전달, 키(a, b, c)가 자동으로 문자열이 된다. 키 제약 : 식별자만 가능함(숫자나 특수문자는 불가능함)
values = d.values()
values

dict_values([10, 20, 30])

In [46]:
d = dict({'a':10, 'b':20, 'c':30})
#d = {'a': 10, 'b': 20, 'c': 30} # 자체가 딕셔너리 이므로, dict() 감싸는 것이 불필요함.
values = d.values()
values


dict_values([10, 20, 30])

#### 지능형 딕셔너리 

In [ ]:
# 지능형 딕셔너리 (Dict Comprehension) 예제

# 1. 기본적인 지능형 딕셔너리
squares = {x: x**2 for x in range(1, 6)}
print("제곱 딕셔너리:", squares)

# 2. 조건이 있는 지능형 딕셔너리
even_squares = {x: x**2 for x in range(1, 11) if x % 2 == 0}
print("짝수 제곱 딕셔너리:", even_squares)

# 3. 문자열 처리
word_lengths = {word: len(word) for word in ['apple', 'banana', 'cherry']}
print("단어 길이 딕셔너리:", word_lengths)

# 4. 딕셔너리에서의 키-값 변환
original_dict = {'a': 1, 'b': 2, 'c': 3}
inverted_dict = {value: key for key, value in original_dict.items()}
# value에는 key값을 넣어라 : original_dict에서 가져온 key와 value 갑 중에서 (260519) 


print("역전 딕셔너리:", inverted_dict)

# 5. 조건부 값 설정
temperature_fahrenheit = {'New York': 70, 'London': 65, 'Tokyo': 80}
temperature_celsius = {city: (temp - 32) * 5/9 for city, temp in temperature_fahrenheit.items()}
print("섭씨 온도:", temperature_celsius)

제곱 딕셔너리: {1: 1, 2: 4, 3: 9, 4: 16, 5: 25}
짝수 제곱 딕셔너리: {2: 4, 4: 16, 6: 36, 8: 64, 10: 100}
단어 길이 딕셔너리: {'apple': 5, 'banana': 6, 'cherry': 6}
역전 딕셔너리: {1: 'a', 2: 'b', 3: 'c'}
섭씨 온도: {'New York': 21.11111111111111, 'London': 18.333333333333332, 'Tokyo': 26.666666666666668}


In [ ]:
inverted_dict[1] # key와 value가 서로 교환

'a'